# Experiment series — Predictive accuracy versus causal effect estimation (JAX)

**Scientific question:** Can models that predict observational outcomes accurately nevertheless give incorrect intervention-effect estimates?  
**Design:** Six causal scenarios (12 configurations), held-out evaluation, sample-size study, targeted cross-environment transfer, explicit observed/oracle separation.

**Important:** Regression-based plug-in contrasts are causal only when the required adjustment set is observed and causal assumptions hold. In hidden-confounder and noisy-proxy scenarios, plug-in contrasts are **not identified causal effects**; their errors relative to simulator truth are diagnostic only. We do not train on oracle outcomes or hidden variables.

**Run locally:** `pip install -U "jax[cpu]" numpy pandas matplotlib jupyter` then **Restart & Run All**.  Results saved next to notebook under `causal_model_comparison_outputs/`. Prior M1 CSVs are consumed if found; otherwise the *exact same* JAX SCM and documented seeds regenerate M1-equivalent observations in memory. This notebook does not overwrite Synthetic benchmark files.

## 1. Reproducibility, data loading and output folders
JAX is used for RNG, regression matrix algebra and nonlinear MLP optimization. Pandas and Matplotlib are used for I/O and figures. Prediction metrics are calculated directly with JAX; is not required.

In [1]:
from pathlib import Path
import json, warnings, platform, time
import numpy as np  # CPU interchange with pandas/matplotlib only; JAX does modeling
import pandas as pd
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
# JAX-native metrics; no binary dependency.
def mean_squared_error(y_true, y_pred):
    actual = jnp.asarray(y_true, dtype=jnp.float64)
    estimated = jnp.asarray(y_pred, dtype=jnp.float64)
    return float(jnp.mean((actual-estimated)**2))

def r2_score(y_true, y_pred):
    actual = jnp.asarray(y_true, dtype=jnp.float64)
    estimated = jnp.asarray(y_pred, dtype=jnp.float64)
    denom = jnp.sum((actual-jnp.mean(actual))**2)
    return float(1.-jnp.sum((actual-estimated)**2)/denom) if float(denom)>0 else float('nan')

jax.config.update("jax_enable_x64", True)
OUT = Path("causal_model_comparison_outputs")
for folder in (OUT, OUT / "figures", OUT / "reports", OUT / "predictions"):
    folder.mkdir(parents=True, exist_ok=True)
INPUT_ROOT_CANDIDATES = [Path("jax_outputs/advanced"), Path("../jax_outputs/advanced"), Path("advanced")]
M1_ROOT = next((q for q in INPUT_ROOT_CANDIDATES if (q / "dataset_manifest.csv").exists()), None)
print("JAX",jax.__version__,"backend",jax.default_backend(),"Input data:",M1_ROOT or "not found — deterministic JAX fallback")


JAX 0.9.0.1 backend cpu Input data: jax_outputs/advanced


## 2. Define the experimental scenarios and their true causal effects

For the baseline mechanisms, $Y=2X+1.5U+0.5XU+\varepsilon_Y$. When `nonlinear=True`, add $0.3X^2+0.6\sin X$. The intervention contrast $Y(1)-Y(0)$ is $2+0.5U$ plus $0.3+0.6\sin(1)$ for the nonlinear setting. Its population average is $2+0.5\mu_U$ plus the nonlinear addition.

**Causal targets:** ATE for setting speed 0→1; CATE at each unit's *true* U only for oracle evaluation. In S3 and S4 no valid adjustment on U is available to the learner; S5 has theoretical positivity but weak practical overlap.

In [2]:
CONFIGS = [
 dict(name="S0_baseline__A",scenario="S0_baseline",env="A",mu=0.,sd_u=1.,sd_x=1.,sd_w=0.,nonlinear=False),
 dict(name="S1_nonlinear__A",scenario="S1_nonlinear",env="A",mu=0.,sd_u=1.,sd_x=1.,sd_w=0.,nonlinear=True),
 *[dict(name=f"S2_shift__{e}",scenario="S2_shift",env=e,mu=mu,sd_u=su,sd_x=1.,sd_w=0.,nonlinear=False) for e,mu,su in [('A',0.,1.),('B',2.,1.),('C',0.,2.)]],
 dict(name="S3_hidden__A",scenario="S3_hidden",env="A",mu=0.,sd_u=1.,sd_x=1.,sd_w=0.,nonlinear=False),
 *[dict(name=f"S4_proxy__A__w{w:g}",scenario="S4_proxy",env="A",mu=0.,sd_u=1.,sd_x=1.,sd_w=w,nonlinear=False) for w in [0.,0.5,1.5]],
 *[dict(name=f"S5_overlap__A__x{sx:g}",scenario="S5_overlap",env="A",mu=0.,sd_u=1.,sd_x=sx,sd_w=0.,nonlinear=False) for sx in [1.,0.3,0.05]],
]
assert len(CONFIGS)==12

def pop_ate(cfg):
    return 2. + 0.5*cfg['mu'] + (0.3 + 0.6*np.sin(1.) if cfg['nonlinear'] else 0.)

def true_effect(u,cfg):
    return 2. + 0.5*jnp.asarray(u) + (0.3 + 0.6*jnp.sin(1.) if cfg['nonlinear'] else 0.)

def fallback_data(cfg,n,split):
    # Deterministic JAX fallback seeds, used when exported benchmark files are unavailable.
    ci=next(i for i,c in enumerate(CONFIGS) if c['name']==cfg['name'])
    seed = 410000 + 1000*ci + {500:0,2000:100,10000:200}[n] + {'train':0,'validation':1,'test':2}[split]
    ku,kx,ky,kw=jax.random.split(jax.random.PRNGKey(seed),4)
    u=cfg['mu']+cfg['sd_u']*jax.random.normal(ku,(n,))
    x=0.8*u+cfg['sd_x']*jax.random.normal(kx,(n,))
    ey=jax.random.normal(ky,(n,)); w=u+cfg['sd_w']*jax.random.normal(kw,(n,))
    def g(z):
        res=2*z+1.5*u+0.5*z*u+ey
        if cfg['nonlinear']: res=res+0.3*z*z+0.6*jnp.sin(z)
        return res
    oracle=pd.DataFrame({"U":np.asarray(u),"X":np.asarray(x),"Y":np.asarray(g(x)),"W":np.asarray(w),"Y0":np.asarray(g(jnp.zeros_like(x))),"Y1":np.asarray(g(jnp.ones_like(x))),"ITE":np.asarray(g(jnp.ones_like(x))-g(jnp.zeros_like(x)))})
    cols= ['X','Y'] if cfg['scenario']=='S3_hidden' else (['W','X','Y'] if cfg['scenario']=='S4_proxy' else ['U','X','Y'])
    return oracle[cols].copy(),oracle

manifest=pd.read_csv(M1_ROOT/'dataset_manifest.csv') if M1_ROOT else None

def load_split(cfg,n,split):
    if manifest is not None:
        m=manifest[(manifest['config']==cfg['name'])&(manifest['n']==n)&(manifest['split']==split)]
        if len(m)==1:
            entry=m.iloc[0]
            def locate(path):
                path=Path(path)
                for candidate in [path, M1_ROOT/path.name, M1_ROOT/'data'/path.name, M1_ROOT.parent.parent/path]:
                    if candidate.exists(): return candidate
                return None
            obs=locate(entry['observed_file']); oracle=locate(entry['oracle_file'])
            if obs and oracle: return pd.read_csv(obs),pd.read_csv(oracle)
    return fallback_data(cfg,n,split)

for c in CONFIGS:
    d,o=load_split(c,500,'test')
    assert len(d)==500 and set(d.columns).issubset(set(o.columns))
    assert np.allclose(o['ITE'],np.asarray(true_effect(o['U'],c)),rtol=1e-5,atol=1e-5)
print('Loaded and checked all 12 configurations; population ATEs:')
print(pd.DataFrame([{'config':c['name'],'ATE':pop_ate(c),'learner_columns':','.join(load_split(c,500,'test')[0].columns)} for c in CONFIGS]).to_string(index=False))


Loaded and checked all 12 configurations; population ATEs:
              config      ATE learner_columns
      S0_baseline__A 2.000000           U,X,Y
     S1_nonlinear__A 2.804883           U,X,Y
         S2_shift__A 2.000000           U,X,Y
         S2_shift__B 3.000000           U,X,Y
         S2_shift__C 2.000000           U,X,Y
        S3_hidden__A 2.000000             X,Y
     S4_proxy__A__w0 2.000000           W,X,Y
   S4_proxy__A__w0.5 2.000000           W,X,Y
   S4_proxy__A__w1.5 2.000000           W,X,Y
   S5_overlap__A__x1 2.000000           U,X,Y
 S5_overlap__A__x0.3 2.000000           U,X,Y
S5_overlap__A__x0.05 2.000000           U,X,Y


## 3. Four JAX regression baselines and their estimands

1. **Speed-only linear**: $Y\sim1+X$. Observational association; *not* automatically causal.  
2. **Adjusted additive linear**: $Y\sim1+X+Z$, with $Z=U$ when observed or $W$ in proxy scenario.  
3. **Adjusted interaction**: $Y\sim1+X+Z+XZ$. Correct functional form for S0 and S2 when $Z=U$.  
4. **Flexible nonlinear features**: adds $X^2,\sin X$ to interaction model; contains S1's true response functional form if $U$ observed.

S3 can only fit speed-only. In S4 we do not have U, so models adjust for W and their causal validity is **not guaranteed**. We standardize neither x nor z so the do(0) / do(1) contrasts are easy to interpret. A very small ridge penalty only prevents singular matrices.

In [3]:
def get_adjuster(df):
    return 'U' if 'U' in df else ('W' if 'W' in df else None)

def design(df,kind,x_override=None):
    x=jnp.asarray(df['X'].to_numpy() if x_override is None else np.full(len(df),x_override),dtype=jnp.float64)
    parts=[jnp.ones_like(x),x]
    if kind!='speed_only':
        zcol=get_adjuster(df)
        if zcol is None: raise ValueError('Confounder not observed for adjustment')
        z=jnp.asarray(df[zcol].to_numpy(),dtype=jnp.float64)
        parts.extend([z])
        if kind in ('interaction','nonlinear'): parts.append(x*z)
        if kind=='nonlinear': parts.extend([x*x,jnp.sin(x)])
    return jnp.column_stack(parts)

def fit_regression(train,kind):
    a=design(train,kind)
    y=jnp.asarray(train['Y'].to_numpy(),dtype=jnp.float64)
    lam=1e-6*jnp.eye(a.shape[1]); lam=lam.at[0,0].set(0.)
    return jnp.linalg.solve(a.T@a+lam,a.T@y)

def predict_regression(beta,df,kind,x_override=None):
    return np.asarray(design(df,kind,x_override)@beta)

def contrast_regression(beta,df,kind):
    return predict_regression(beta,df,kind,1.)-predict_regression(beta,df,kind,0.)


## 4. Main experiment — independent train/validation/test splits across 12 configurations

Every result is evaluated on an independent test split. Metrics: test RMSE, $R^2$, estimated ATE (g-computation-style plug-in contrast), ATE absolute error, and CATE RMSE against the simulator's individual oracle effect. CATE RMSE is a *diagnostic per-unit effect error*, not a guarantee that an estimator identifies conditional treatment effects. Labels explicitly mark identification status. Validation split is reserved for model selection in the later neural experiment.

In [4]:
records=[]; unit_records=[]
for cfg in CONFIGS:
    train,_=load_split(cfg,2000,'train')
    val,_=load_split(cfg,2000,'validation')
    test,oracle=load_split(cfg,2000,'test')
    identification='not identified with observed variables' if cfg['scenario'] in ('S3_hidden','S4_proxy') and not (cfg['scenario']=='S4_proxy' and cfg['sd_w']==0) else 'backdoor identifiable given U (or W=U)'
    kinds=['speed_only'] + (['additive','interaction','nonlinear'] if get_adjuster(train) else [])
    for kind in kinds:
        beta=fit_regression(train,kind)
        prediction=predict_regression(beta,test,kind)
        effect=contrast_regression(beta,test,kind)
        target=np.asarray(true_effect(oracle['U'].to_numpy(),cfg))
        records.append(dict(config=cfg['name'],scenario=cfg['scenario'],model=kind,n_train=len(train),test_rmse=float(np.sqrt(mean_squared_error(test['Y'],prediction))),test_r2=float(r2_score(test['Y'],prediction)),ate_hat=float(effect.mean()),true_ate=pop_ate(cfg),ate_abs_error=float(abs(effect.mean()-pop_ate(cfg))),cate_rmse=float(np.sqrt(np.mean((effect-target)**2))),identification=identification))
        if cfg['name'] in ['S0_baseline__A','S1_nonlinear__A','S3_hidden__A','S4_proxy__A__w1.5']:
            unit_records.append(pd.DataFrame({'config':cfg['name'],'model':kind,'x':test['X'],'y':test['Y'],'prediction':prediction,'effect_hat':effect,'oracle_effect':target}))
results=pd.DataFrame(records).sort_values(['config','model']).reset_index(drop=True)
results.to_csv(OUT/'reports'/'model_comparison_all_scenarios.csv',index=False)
if unit_records: pd.concat(unit_records,ignore_index=True).to_csv(OUT/'predictions'/'selected_test_predictions.csv',index=False)
print('Evaluated',len(results),'model/scenario combinations')
print(results[['config','model','test_rmse','ate_hat','true_ate','ate_abs_error','cate_rmse']].round(3).to_string(index=False))


Evaluated 45 model/scenario combinations
              config       model  test_rmse  ate_hat  true_ate  ate_abs_error  cate_rmse
      S0_baseline__A    additive      1.251    2.087     2.000          0.087      0.499
      S0_baseline__A interaction      1.014    2.058     2.000          0.058      0.054
      S0_baseline__A   nonlinear      1.015    2.015     2.000          0.015      0.012
      S0_baseline__A  speed_only      1.718    2.819     2.000          0.819      0.951
     S1_nonlinear__A    additive      1.705    2.268     2.805          0.537      0.740
     S1_nonlinear__A interaction      1.099    2.281     2.805          0.524      0.632
     S1_nonlinear__A   nonlinear      0.974    2.795     2.805          0.010      0.014
     S1_nonlinear__A  speed_only      2.086    2.993     2.805          0.188      0.539
         S2_shift__A    additive      1.267    2.006     2.000          0.006      0.488
         S2_shift__A interaction      0.999    2.001     2.000       

## 5. Diagnostic plots — predictive error and causal-effect error

Interpret each scenario separately: S3 and noisy S4 deliberately lack an identified estimand under the learner's observed-data adjustment assumptions. Low prediction error cannot turn these contrasts into identified causal effects.

In [5]:
fig,axes=plt.subplots(1,2,figsize=(15,5))
for m,g in results.groupby('model'):
    axes[0].scatter(g['test_rmse'],g['ate_abs_error'],label=m,alpha=.8)
axes[0].set(xlabel='Test RMSE (prediction)',ylabel='Absolute error in simulator ATE',title='Predictive versus causal-effect accuracy')
axes[0].legend(fontsize=8); axes[0].grid(alpha=.2)
pivot=results.pivot(index='config',columns='model',values='ate_abs_error').sort_index()
width=.8/max(1,len(pivot.columns)); xx=np.arange(len(pivot))
for j,col in enumerate(pivot.columns):axes[1].bar(xx-.4+(j+.5)*width,pivot[col].fillna(0),width,label=col)
axes[1].set_xticks(xx, pivot.index,rotation=70,ha='right',fontsize=7)
axes[1].set(ylabel='ATE absolute error',title='Effect estimation error by model/scenario')
axes[1].legend(fontsize=7); axes[1].grid(axis='y',alpha=.2)
fig.tight_layout();fig.savefig(OUT/'figures'/'01_prediction_vs_causal_error.png',dpi=165,bbox_inches='tight');plt.show()


/tmp/ipykernel_1635/2498237644.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout();fig.savefig(OUT/'figures'/'01_prediction_vs_causal_error.png',dpi=165,bbox_inches='tight');plt.show()


## 5A. Expanded diagnostics: prediction quality by scenario

Compare error distributions across structural mechanisms. Errors for hidden confounding and noisy proxies are descriptive and must not be interpreted as identified effects.

In [6]:
# Prediction RMSE and R² as scenario-by-model matrices.
fig, axes = plt.subplots(1, 2, figsize=(17, 6))
for ax, metric, ttl in zip(axes, ['test_rmse','test_r2'], ['Prediction RMSE','Out-of-sample R²']):
    p=results.pivot(index='config',columns='model',values=metric).sort_index()
    vals=p.to_numpy(dtype=float)
    im=ax.imshow(np.ma.masked_invalid(vals),aspect='auto',cmap='viridis')
    ax.set_xticks(range(len(p.columns)),p.columns,rotation=35,ha='right')
    ax.set_yticks(range(len(p.index)),p.index,fontsize=8)
    ax.set_title(ttl)
    fig.colorbar(im,ax=ax,fraction=.04,pad=.02)
fig.tight_layout()
fig.savefig(OUT/'figures'/'04_prediction_metric_heatmaps.png',dpi=160,bbox_inches='tight')
plt.show()


/tmp/ipykernel_1635/4145273687.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5B. Effect estimation: bias and heterogeneity errors

Signed ATE error and CATE RMSE diagnose different aspects of model performance. These quantities are measured against simulator truth even in nonidentified settings.

In [7]:
fig,axes=plt.subplots(1,2,figsize=(17,6))
for ax,metric,title in zip(axes,['signed_ate_error','cate_rmse'],['Signed ATE estimation error','CATE RMSE vs simulator truth']):
    rr=results.assign(signed_ate_error=results.ate_hat-results.true_ate)
    p=rr.pivot(index='config',columns='model',values=metric)
    for k,m in enumerate(p.columns):
        xx=np.arange(len(p))+((k-(len(p.columns)-1)/2)*.15)
        ax.scatter(xx,p[m],label=m,s=32)
    ax.set_xticks(np.arange(len(p)),p.index,rotation=70,ha='right',fontsize=8)
    ax.set_title(title);ax.grid(axis='y',alpha=.2)
    if metric=='signed_ate_error':ax.axhline(0,color='black',linewidth=.9)
    ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(OUT/'figures'/'05_effect_error_diagnostics.png',dpi=160,bbox_inches='tight')
plt.show()


/tmp/ipykernel_1635/985985990.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5C. Observational predictions versus intervention responses

Use a shared evaluation sample to compare plug-in response curves. These are causal estimands only for models with sufficient adjustment and compatible assumptions.

In [8]:
cfg=CONFIGS[0]
train,_=load_split(cfg,2000,'train')
test,oracle=load_split(cfg,2000,'test')
xs=np.linspace(-2,2,81)
fig,ax=plt.subplots(figsize=(9,5))
for kind in ['speed_only','additive','interaction','nonlinear']:
    beta=fit_regression(train,kind)
    curve=[float(predict_regression(beta,test,kind,float(z)).mean()) for z in xs]
    ax.plot(xs,curve,label=kind)
ax.plot(xs,2*xs,'k--',linewidth=2,label='True do(X=x) mean')
ax.set(xlabel='Set speed x',ylabel='Mean energy response',title='Estimated response curves vs ground truth (baseline)')
ax.legend();ax.grid(alpha=.2)
fig.tight_layout();fig.savefig(OUT/'figures'/'06_response_curves.png',dpi=165);plt.show()


/tmp/ipykernel_1635/3489056011.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout();fig.savefig(OUT/'figures'/'06_response_curves.png',dpi=165);plt.show()


## 5D. Unit-level treatment effects and calibration

The identity line represents perfect recovery of simulator ITE. Each point is an estimated plug-in contrast for a held-out unit.

In [9]:
cfg=CONFIGS[0]
train,_=load_split(cfg,2000,'train');test,oracle=load_split(cfg,2000,'test')
truth=np.asarray(true_effect(oracle['U'].to_numpy(),cfg))
fig,axes=plt.subplots(1,2,figsize=(13,5))
for kind in ['speed_only','additive','interaction','nonlinear']:
    b=fit_regression(train,kind); e=contrast_regression(b,test,kind)
    axes[0].scatter(truth[::12],e[::12],s=13,alpha=.3,label=kind)
    axes[1].hist(np.asarray(e),bins=np.linspace(float(min(truth.min(),2)-2),float(max(truth.max(),4)+2),40),alpha=.3,density=True,label=kind)
axes[0].plot([truth.min(),truth.max()],[truth.min(),truth.max()],'k--')
axes[0].set(xlabel='True individual effect',ylabel='Estimated individual effect',title='Unit-level recovery')
axes[1].hist(truth,bins=35,histtype='step',color='black',linewidth=2,density=True,label='Oracle effect')
axes[1].set(xlabel='Treatment effect',ylabel='Density',title='Effect distribution')
for ax in axes:ax.legend(fontsize=8);ax.grid(alpha=.2)
fig.tight_layout();fig.savefig(OUT/'figures'/'07_unit_effects.png',dpi=160);plt.show()


/tmp/ipykernel_1635/1687386562.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout();fig.savefig(OUT/'figures'/'07_unit_effects.png',dpi=160);plt.show()


## 5E. Confounder observability and proxy noise

A noisy proxy does not generally block the backdoor path. Compare effect estimates across proxy quality while keeping the underlying causal estimand fixed.

In [10]:
subset=results[results.config.str.startswith('S4_proxy')]
fig,ax=plt.subplots(figsize=(9,5))
for name,g in subset.groupby('model'):
    g=g.assign(proxy_sd=g.config.str.extract(r'w([0-9.]+)$')[0].astype(float)).sort_values('proxy_sd')
    ax.plot(g.proxy_sd,g.ate_hat,marker='o',label=name)
ax.axhline(2,color='black',linestyle='--',label='True ATE')
ax.set(xlabel='Proxy noise standard deviation',ylabel='Estimated effect',title='Measurement error and confounding')
ax.legend();ax.grid(alpha=.2)
fig.tight_layout();fig.savefig(OUT/'figures'/'08_proxy_sensitivity.png',dpi=165);plt.show()


/tmp/ipykernel_1635/1321472431.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout();fig.savefig(OUT/'figures'/'08_proxy_sensitivity.png',dpi=165);plt.show()


## 5F. Practical positivity and treatment overlap

Every configuration has positive Gaussian treatment density for every finite speed, but extreme conditioning can leave little overlap in observed samples.

In [11]:
fig,axes=plt.subplots(1,3,figsize=(15,4),sharex=True,sharey=True)
for ax,sd in zip(axes,[1.,.3,.05]):
    cfg=next(c for c in CONFIGS if c['name']==f'S5_overlap__A__x{sd:g}')
    d,_=load_split(cfg,2000,'test')
    ax.scatter(d['U'][::3],d['X'][::3],s=9,alpha=.3)
    ax.set_title(f'Assignment noise σX={sd:g}')
    ax.set_xlabel('Workload U');ax.grid(alpha=.2)
axes[0].set_ylabel('Observed speed X')
fig.tight_layout();fig.savefig(OUT/'figures'/'09_overlap_scatters.png',dpi=165);plt.show()


/tmp/ipykernel_1635/2492233331.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout();fig.savefig(OUT/'figures'/'09_overlap_scatters.png',dpi=165);plt.show()


## 6. Sample size sensitivity — same baseline and fixed test design

Fit the correct interaction model and the speed-only association model on 500, 2,000 and 10,000 training samples, using their corresponding independent test datasets. This helps separate finite-sample variation from systematic confounding bias.

In [12]:
size_rows=[]
cfg=CONFIGS[0]
for nn in [500,2000,10000]:
    train,_=load_split(cfg,nn,'train'); test,oracle=load_split(cfg,nn,'test')
    for model in ['speed_only','interaction']:
        b=fit_regression(train,model); eff=contrast_regression(b,test,model)
        size_rows.append(dict(n=nn,model=model,rmse=float(np.sqrt(mean_squared_error(test['Y'],predict_regression(b,test,model)))),ate_hat=float(eff.mean()),ate_abs_error=float(abs(eff.mean()-pop_ate(cfg)))))
size_df=pd.DataFrame(size_rows);size_df.to_csv(OUT/'reports'/'sample_size_sensitivity.csv',index=False)
fig,ax=plt.subplots(figsize=(7,4))
for name,g in size_df.groupby('model'):
    ax.plot(g['n'],g['ate_abs_error'],marker='o',label=name)
ax.set(xscale='log',xlabel='Training observations',ylabel='ATE absolute error',title='Sample size cannot remove confounding bias by itself');ax.legend();ax.grid(alpha=.2)
fig.tight_layout();fig.savefig(OUT/'figures'/'02_sample_size_sensitivity.png',dpi=170);plt.show()
print(size_df.round(4).to_string(index=False))


    n       model   rmse  ate_hat  ate_abs_error
  500  speed_only 1.6820   2.7841         0.7841
  500 interaction 0.9922   2.0968         0.0968
 2000  speed_only 1.7177   2.8187         0.8187
 2000 interaction 1.0144   2.0580         0.0580
10000  speed_only 1.7104   2.7576         0.7576
10000 interaction 0.9979   2.0037         0.0037


/tmp/ipykernel_1635/1566484183.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout();fig.savefig(OUT/'figures'/'02_sample_size_sensitivity.png',dpi=170);plt.show()


## 7. Cross-environment transport — train A, test B and C

The structural mechanisms remain invariant in S2. The true **CATE function** depends on U, while the **ATE** changes with the target environment's workload distribution. A causal g-computation plug-in must average over the **target population's U**, not the training population's U. We evaluate on the held-out target-environment data and do not assume the baseline speed-only regression transports causally.

In [13]:
ca=next(c for c in CONFIGS if c['name']=='S2_shift__A')
train_a,_=load_split(ca,2000,'train')
transfer=[]
for model in ['speed_only','interaction']:
    b=fit_regression(train_a,model)
    for env in ['A','B','C']:
        ct=next(c for c in CONFIGS if c['name']==f'S2_shift__{env}')
        test,_=load_split(ct,2000,'test')
        contrast=contrast_regression(b,test,model)
        transfer.append(dict(model=model,train_env='A',test_env=env,true_ate=pop_ate(ct),ate_hat=float(contrast.mean()),ate_abs_error=float(abs(contrast.mean()-pop_ate(ct))),predictive_rmse=float(np.sqrt(mean_squared_error(test['Y'],predict_regression(b,test,model))))))
transfer_df=pd.DataFrame(transfer);transfer_df.to_csv(OUT/'reports'/'environment_transport.csv',index=False)
print(transfer_df.round(4).to_string(index=False))
fig,ax=plt.subplots(figsize=(7,4))
for m,g in transfer_df.groupby('model'):
    ax.plot(g['test_env'],g['ate_hat'],marker='o',label=m)
ax.plot(['A','B','C'],[2.,3.,2.],color='black',linestyle='--',label='Oracle population ATE')
ax.set(xlabel='Evaluation environment',ylabel='ATE',title='Transporting from A to target population B/C');ax.legend();ax.grid(alpha=.2)
fig.tight_layout();fig.savefig(OUT/'figures'/'03_cross_environment_ate.png',dpi=170);plt.show()


      model train_env test_env  true_ate  ate_hat  ate_abs_error  predictive_rmse
 speed_only         A        A       2.0   2.7567         0.7567           1.7193
 speed_only         A        B       3.0   2.7567         0.2433           4.3887
 speed_only         A        C       2.0   2.7567         0.7567           3.3884
interaction         A        A       2.0   2.0009         0.0009           0.9990
interaction         A        B       3.0   2.9877         0.0123           1.0202
interaction         A        C       2.0   1.9912         0.0088           1.0002


/tmp/ipykernel_1635/1812318087.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout();fig.savefig(OUT/'figures'/'03_cross_environment_ate.png',dpi=170);plt.show()


## 8. Optional nonlinear JAX neural model (MLP) with validation-based early stopping

This model illustrates that flexible supervised prediction is **not automatically causal identification**. Trained only with learner-visible features. To keep laptop execution predictable, run it on two core scenarios (S0 and S1), 2,000 observations each. Its stochastic training is JAX-seeded; validation data are separate from test data. Early stopping selects among checkpoints. The neural plug-in effect is meaningful as an adjustment-based estimate only when U is observed and correct causal assumptions hold.

In [14]:
def network_features(df,x_override=None):
    x=jnp.asarray(df['X'].to_numpy() if x_override is None else np.full(len(df),x_override),dtype=jnp.float64)
    zcol=get_adjuster(df)
    z=jnp.asarray(df[zcol].to_numpy(),dtype=jnp.float64) if zcol else jnp.zeros_like(x)
    return jnp.column_stack([x,z])

def init_mlp(key,width=24):
    k1,k2=jax.random.split(key)
    return [(jax.random.normal(k1,(2,width))*jnp.sqrt(1.),jnp.zeros(width)),
            (jax.random.normal(k2,(width,1))*jnp.sqrt(2./width),jnp.zeros(1))]

def forward(params,x):
    w1,b1=params[0];w2,b2=params[1]
    return (jax.nn.tanh(x@w1+b1)@w2+b2).ravel()

@jax.jit
def step_mlp(params,x,y,lr):
    def loss(pr): return jnp.mean((forward(pr,x)-y)**2)
    gradients=jax.grad(loss)(params)
    return jax.tree_util.tree_map(lambda a,g:a-lr*g,params,gradients)

neural_rows=[]
for i,cn in enumerate(['S0_baseline__A','S1_nonlinear__A']):
    cfg=next(c for c in CONFIGS if c['name']==cn)
    train,_=load_split(cfg,2000,'train');val,_=load_split(cfg,2000,'validation');test,oracle=load_split(cfg,2000,'test')
    # Only covariates are standardized; treatment interventions are transformed using the same training scale.
    tr=network_features(train); vv=network_features(val)
    m=jnp.mean(tr,axis=0); s=jnp.maximum(jnp.std(tr,axis=0),1e-6)
    tx=(tr-m)/s; vx=(vv-m)/s
    ty=jnp.asarray(train['Y'].to_numpy());vy=jnp.asarray(val['Y'].to_numpy())
    ym=ty.mean();ys=jnp.maximum(ty.std(),1e-6);ty=(ty-ym)/ys
    params=init_mlp(jax.random.PRNGKey(890+i))
    best=params;best_loss=float('inf');best_epoch=0
    for epoch in range(700):
        params=step_mlp(params,tx,ty,0.018)
        if epoch%20==0:
            vloss=float(jnp.mean((forward(params,vx)-((vy-ym)/ys))**2))
            if vloss<best_loss:best_loss=vloss;best=params;best_epoch=epoch
    pred=lambda d,x=None:np.asarray(forward(best,(network_features(d,x)-m)/s)*ys+ym)
    yy=pred(test);effects=pred(test,1.)-pred(test,0.)
    true=np.asarray(true_effect(oracle['U'].to_numpy(),cfg))
    neural_rows.append(dict(config=cn,model='jax_mlp',best_epoch=best_epoch,test_rmse=float(np.sqrt(mean_squared_error(test['Y'],yy))),test_r2=float(r2_score(test['Y'],yy)),ate_hat=float(effects.mean()),true_ate=pop_ate(cfg),ate_abs_error=float(abs(effects.mean()-pop_ate(cfg))),cate_rmse=float(np.sqrt(np.mean((effects-true)**2)))))
mlp_df=pd.DataFrame(neural_rows);mlp_df.to_csv(OUT/'reports'/'jax_mlp_evaluation.csv',index=False)
print(mlp_df.round(4).to_string(index=False))


         config   model  best_epoch  test_rmse  test_r2  ate_hat  true_ate  ate_abs_error  cate_rmse
 S0_baseline__A jax_mlp         680     1.2280   0.8990   2.5533    2.0000         0.5533     0.8009
S1_nonlinear__A jax_mlp         680     1.3583   0.9108   3.5389    2.8049         0.7340     0.8867


## 8A. Prediction residual patterns

Residuals against speed can reveal systematic predictive misspecification; this diagnostic alone cannot establish a causal model.

In [15]:
cfg=CONFIGS[1]
train,_=load_split(cfg,2000,'train');test,_=load_split(cfg,2000,'test')
fig,axes=plt.subplots(1,3,figsize=(15,4),sharey=True)
for ax,kind in zip(axes,['speed_only','interaction','nonlinear']):
    b=fit_regression(train,kind);pred=predict_regression(b,test,kind)
    ax.scatter(test['X'][::5],(test['Y'].to_numpy()-pred)[::5],s=10,alpha=.3)
    ax.axhline(0,c='black',linestyle='--',linewidth=1)
    ax.set(xlabel='Observed speed',title=kind);ax.grid(alpha=.2)
axes[0].set_ylabel('Observed minus predicted energy')
fig.suptitle('Residual patterns on nonlinear scenario',y=1.02)
fig.tight_layout();fig.savefig(OUT/'figures'/'10_prediction_residuals.png',dpi=165,bbox_inches='tight');plt.show()


/tmp/ipykernel_1635/746375285.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout();fig.savefig(OUT/'figures'/'10_prediction_residuals.png',dpi=165,bbox_inches='tight');plt.show()


## 9. Scientific checks, interpretation, and limitations

**Predicted theoretical behaviors** (not guaranteed numerical outcomes):
- On S0/S2, the interaction specification with the observed U should approach the true effect as data increase. The speed-only specification generally has confounding bias.
- S1 needs a nonlinear specification to describe the shape of the intervention response.
- S2 average effects transport only after accounting for the target U distribution.
- S3/S4 (nonzero W-noise): a predictive improvement does not establish identification; observed-data estimation without additional assumptions is not sufficient.
- S5 has positive Gaussian treatment density but may have poor finite-sample overlap and numerical instability.

**Limitations:** Synthetic observations and known generator; one SCM family; linear/polynomial baselines; the JAX MLP is an illustration, not a validated state-of-the-art causal estimator. Assess robustness with repeated seeds and methods for identification/partial-identification in future experiments. No scientific novelty is claimed merely from generating these plots.

In [16]:
assert results['test_rmse'].notna().all()
assert np.isfinite(results[['ate_hat','ate_abs_error','cate_rmse']].to_numpy()).all()
assert (results['test_rmse'] >= 0).all()
assert all(abs(pop_ate(c)-(2+.5*c['mu']+(.3+.6*np.sin(1) if c['nonlinear'] else 0)))<1e-9 for c in CONFIGS)
metadata={'experiment_series':'Predictive and causal estimator evaluation','title':'Predictive and Causal Baselines','jax_version':jax.__version__,'configs':[c['name'] for c in CONFIGS], 'training_n_main':2000,'sizes':[500,2000,10000], 'source':str(M1_ROOT) if M1_ROOT else 'JAX deterministic fallback','number_model_config_results':len(results),'identification_note':'Hidden confounder and noisy proxy effects not identifiable under given observed-data adjustment assumptions','created_files':sorted(str(q) for q in OUT.rglob('*') if q.is_file())}
(OUT/'experiment_metadata.json').write_text(json.dumps(metadata,indent=2))
summary=["# Experiment series — Experiment Summary", "", "## Research question", "Can strong observational predictive accuracy coexist with unreliable estimates of intervention effects?", "", "## Identification caveat", "S3 and S4 with nonzero measurement error are not identified from the observed covariates under the specified adjustment assumptions.", "", "## Main results (observed numerical outputs)", results[['config','model','test_rmse','ate_hat','true_ate','ate_abs_error']].round(4).to_string(index=False), "", "## Cross-environment results", transfer_df.round(4).to_string(index=False), "", "## Interpretation", "These are computational results for a synthetic benchmark, not validation of causal identification in real data. Assess each estimator's assumptions before interpreting a plug-in contrast causally."]
(OUT/'reports'/'scientific_summary.md').write_text('\n'.join(summary))
print('PASS: final consistency checks. Files written to:',OUT.resolve())
for f in sorted(OUT.rglob('*')):
    if f.is_file():print(' ',f, f.stat().st_size,'bytes')


PASS: final consistency checks. Files written to: /mnt/data/causal_model_comparison_outputs
  causal_model_comparison_outputs/experiment_metadata.json 1878 bytes
  causal_model_comparison_outputs/figures/01_prediction_vs_causal_error.png 150038 bytes
  causal_model_comparison_outputs/figures/02_sample_size_sensitivity.png 54725 bytes
  causal_model_comparison_outputs/figures/03_cross_environment_ate.png 75202 bytes
  causal_model_comparison_outputs/figures/04_prediction_metric_heatmaps.png 109627 bytes
  causal_model_comparison_outputs/figures/05_effect_error_diagnostics.png 129022 bytes
  causal_model_comparison_outputs/figures/06_response_curves.png 111875 bytes
  causal_model_comparison_outputs/figures/07_unit_effects.png 119123 bytes
  causal_model_comparison_outputs/figures/08_proxy_sensitivity.png 104588 bytes
  causal_model_comparison_outputs/figures/09_overlap_scatters.png 181475 bytes
  causal_model_comparison_outputs/figures/10_prediction_residuals.png 162743 bytes
  causal_m

## 10. Reproduce on a PC

Install the dependencies:

```bash
pip install -U "jax[cpu]" numpy pandas matplotlib jupyter
```

Put this notebook beside the `jax_outputs/advanced/` folder from Synthetic benchmark (optional). Select **Restart Kernel and Run All**. When source CSVs are absent, the same SCM and seed protocol are generated in memory (fresh samples may differ from previous M1 exports depending on versions/seed mapping), and independent Experiment series results are still saved to `causal_model_comparison_outputs/`. The source dataset manifest is preferred whenever present.

**Files:** `reports/model_comparison_all_scenarios.csv`, `reports/sample_size_sensitivity.csv`, `reports/environment_transport.csv`, `reports/jax_mlp_evaluation.csv`, `reports/scientific_summary.md`, `figures/*.png`, `predictions/*.csv`, `experiment_metadata.json`.

**Next experiment:** Bayesian causal inference and uncertainty (prior/posterior modeling, interval coverage, and sensitivity to misspecification).

All CSV reports, figures, prediction samples, and metadata are saved automatically under `causal_model_comparison_outputs/` beside the notebook. The input dataset folder is optional; a deterministic JAX-generated fallback is provided. **No scikit-learn installation is required.**
